In [1]:
%pip install opencv-pytho+n pyautogui joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: 'opencv-pytho+n': Expected end or semicolon (after name and no valid version specifier)
    opencv-pytho+n
                ^


In [3]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Unable to open the webcam. Check the camera index.")
else:
    print("Webcam opened successfully!")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Unable to receive a frame from the webcam.")
            break

        cv2.imshow("Webcam Test - Press Q to Quit", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
          break

    cap.release()
    cv2.destroyAllWindows()

Webcam opened successfully!


In [5]:
import joblib

model_path = r"D:\ML_project\models\knn_gesture_model.pkl"

knn = joblib.load(model_path)

print("KNN model loaded successfully!")

KNN model loaded successfully!


In [ ]:
print("Available gesture labels:", knn.classes_)

Available gesture labels: ['c' 'down' 'fist' 'fist_moved' 'index' 'l' 'ok' 'palm' 'palm_moved'
 'thumb']


In [ ]:
gesture_actions = {
    "palm": "No action",
    "thumb": "Space key",
    "fist": "Escape key"
}

for gesture, action in gesture_actions.items():
    print(f"{gesture} -> {action}")

palm -> No action
thumb -> Space key
fist -> Escape key


In [7]:
from collections import deque
from collections import Counter

prediction_history = deque(maxlen=5)

def get_stable_gesture(prediction):
    prediction_history.append(str(prediction))

    counts = Counter(prediction_history)
    gesture, count = counts.most_common(1)[0]

    if count >= 4:
        return gesture

    return "Detecting..."

In [ ]:

import cv2
import os

save_path = r"D:\ML_project\dataset\no_hand"
os.makedirs(save_path, exist_ok=True)

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Webcam open nahi ho rahi.")
else:
    count = 0
    target_images = 200

    print("Webcam started!")
    print("Keep the green box empty.")
    print("Press S to save an image.")
    print("Press Q to quit.")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Frame read failed.")
            break

        height, width = frame.shape[:2]
        size = min(400, height, width)

        x1 = (width - size) // 2
        y1 = (height - size) // 2
        x2 = x1 + size
        y2 = y1 + size

        cv2.rectangle(
            frame, (x1, y1), (x2, y2), (0, 255, 0), 2
        )

        cv2.putText(
            frame,
            f"Images: {count}/{target_images}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cv2.imshow("Collect No-Hand Images", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("s"):
            region = frame[y1:y2, x1:x2]
            gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
            resized = cv2.resize(gray, (64, 64))

            file_path = os.path.join(
                save_path, f"no_hand_{count:03d}.jpg"
            )

            if cv2.imwrite(file_path, resized):
                count += 1
                print(f"Saved {count}/{target_images}")

        elif key == ord("q"):
            break

        if count >= target_images:
            break

    cap.release()
    cv2.destroyAllWindows()

    print(f"Collection finished: {count} images saved.")


Webcam started!
Keep the green box empty.
Press S to save an image.
Press Q to quit.
Collection finished: 0 images saved.


In [10]:

import cv2
import numpy as np
import joblib
import pyautogui
from collections import deque, Counter
import time

# Load trained model
model_path = r"D:\ML_project\models\knn_gesture_model.pkl"
knn = joblib.load(model_path)

# Safety: move mouse to top-left corner to trigger failsafe
pyautogui.FAILSAFE = True

# Require a consistent prediction across frames
history = deque(maxlen=5)

# Gesture-to-key mapping
gesture_actions = {
    "palm": "space",
    "fist": "enter",
    "ok": "esc",
    "down": "down",
    "thumb": "up",
    "index": "right",
    "l": "left"
}

cap = cv2.VideoCapture(0)

enabled = False
last_action_gesture = None
last_action_time = 0
cooldown = 1.0

if not cap.isOpened():
    print("Unable to open webcam.")
else:
    print("PC Gesture Control started.")
    print("Press E to ENABLE controls.")
    print("Press D to DISABLE controls.")
    print("Press Q to QUIT.")

    try:
        while True:
            ret, frame = cap.read()

            if not ret:
                break

            height, width = frame.shape[:2]
            size = min(400, height, width)

            x1 = (width - size) // 2
            y1 = (height - size) // 2
            x2, y2 = x1 + size, y1 + size

            # Crop the center region
            region = frame[y1:y2, x1:x2]
            gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)

            # Match training preprocessing
            resized = cv2.resize(
                gray, (64, 64),
                interpolation=cv2.INTER_AREA
            )

            image = resized.astype("float32") / 255.0
            image = image.reshape(1, -1)

            # Predict gesture
            probabilities = knn.predict_proba(image)[0]
            best_index = np.argmax(probabilities)
            confidence = probabilities[best_index]

            prediction = str(knn.classes_[best_index])

            # Only accept unanimous KNN votes
            if confidence >= 0.99:
                history.append(prediction)
            else:
                history.clear()
                last_action_gesture = None

            stable_gesture = None

            if len(history) == 5:
                counts = Counter(history)
                gesture, count = counts.most_common(1)[0]

                if count >= 4:
                    stable_gesture = gesture

            # Execute one action per stable gesture
            now = time.time()

            if (
                enabled
                and stable_gesture in gesture_actions
                and stable_gesture != last_action_gesture
                and now - last_action_time >= cooldown
            ):
                key = gesture_actions[stable_gesture]
                pyautogui.press(key)

                print(f"Gesture: {stable_gesture} -> Key: {key}")

                last_action_gesture = stable_gesture
                last_action_time = now

            # Show status
            status = "ENABLED" if enabled else "DISABLED"
            label = stable_gesture if stable_gesture else "Detecting..."

            cv2.rectangle(
                frame, (x1, y1), (x2, y2), (0, 255, 0), 2
            )

            cv2.putText(
                frame,
                f"Control: {status}",
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Gesture: {label}",
                (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            cv2.imshow("PC Gesture Control", frame)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("e"):
                enabled = True
                history.clear()
                last_action_gesture = None
                print("Controls ENABLED")

            elif key == ord("d"):
                enabled = False
                history.clear()
                last_action_gesture = None
                print("Controls DISABLED")

            elif key == ord("q"):
                break

    except pyautogui.FailSafeException:
        print("PyAutoGUI failsafe triggered. Stopping controls.")

    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("PC Gesture Control stopped.")


PC Gesture Control started.
Press E to ENABLE controls.
Press D to DISABLE controls.
Press Q to QUIT.
Controls ENABLED
Gesture: ok -> Key: esc
Controls ENABLED
Gesture: ok -> Key: esc
PC Gesture Control stopped.
